# RAG Data Ingestion Pipeline (MVP)
Aufbau einer lokalen Vektordatenbank (Chroma) fuer Retrieval-Augmented Generation.

Die Pipeline laedt alle PDFs aus dem Verzeichnis `Rag Database`, zerlegt sie in Chunks,
erzeugt Embeddings ueber Google Gemini und speichert alles in einer persistenten
Chroma-DB unter `./chroma_db`.

**Abhaengigkeiten:** `langchain`, `langchain-community`, `langchain-google-genai`,
`langchain-text-splitters`, `pypdf`, `chromadb`, `python-dotenv`

## 1) SETUP – API Key aus .env laden
Gleicher Mechanismus wie in `agent.ipynb` / `app.py`: `GEMINI_API_KEY` wird ueber
`python-dotenv` geladen und vor der Verwendung validiert.

In [ ]:
import os
from glob import glob
from dotenv import load_dotenv

# API Key aus .env laden (gleicher Ansatz wie in agent.ipynb / app.py)
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY nicht in .env gefunden!")

os.environ["GOOGLE_API_KEY"] = api_key  # fuer langchain-google-genai
print("Setup abgeschlossen. GEMINI_API_KEY geladen.")

## 2) DOCUMENT LOADING – PDFs dynamisch aus `Rag Database` laden
Alle `*.pdf`-Dateien im Verzeichnis werden automatisch gefunden und mit `PyPDFLoader`
seitenweise geladen. Jede Seite wird zu einem eigenen `Document` inkl. Metadaten
(Quellpfad, Seitenzahl).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

DATA_DIR = "Rag Database"
pdf_files = sorted(glob(os.path.join(DATA_DIR, "*.pdf")))

if not pdf_files:
    raise FileNotFoundError(f"Keine PDFs im Verzeichnis '{DATA_DIR}' gefunden!")

documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    documents.extend(loader.load())

print(f"Gefundene PDFs:      {len(pdf_files)}")
for p in pdf_files:
    print(f"  - {p}")
print(f"Geladene Seiten (Documents): {len(documents)}")

## 3) CHUNKING – Text mit RecursiveCharacterTextSplitter zerlegen
Der rekursive Splitter teilt den Text entlang natuerlicher Trennzeichen (Absatz, Satz,
Wort), damit zusammenhaengende Inhalte moeglichst in einem Chunk bleiben.

- `chunk_size = 1000` (Zeichen pro Chunk)
- `chunk_overlap = 200` (Ueberlappung, damit Kontext am Rand erhalten bleibt)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)
print(f"Erstellte Chunks: {len(chunks)}")
print(f"Beispiel-Metadaten eines Chunks: {chunks[0].metadata}")

## 4) EMBEDDINGS & VECTOR STORE – Chroma-Datenbank erstellen und persistieren
Embeddings ueber Googles Modell `models/gemini-embedding-2`.

**Hinweis zum Free-Tier-Quota:** Das Embedding-Kontingent betraegt 100 Requests/Min.
Bei einem geteilten/umschlagnen API-Key ist dieses Fenster oft kurzfristig voll,
sodass `429 RESOURCE_EXHAUSTED` auftreten kann. Daher werden die Chunks hier
chargenweise mit Retry + Backoff eingefuegt – das ist robuster als ein einzelner
`from_documents`-Aufruf. Die fertige DB liegt in `./chroma_db` und kann spaeter ohne
erneutes Embedding geladen werden.

**Idempotenz:** Vor dem Einfuegen wird eine bestehende Collection ueber
`delete_collection()` geloescht, damit mehrfaches Ausfuehren des Notebooks die
Vektoren nicht dupliziert (robuster unter Windows als ein Verzeichnis-Loeschung).

In [ ]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

CHROMA_DIR = "./chroma_db"
BATCH_SIZE = 100  # Googles Embedding-API erlaubt max. 100 Texte pro Request
MAX_TRIES = 6     # Retry-Versuche pro Batch bei 429 (Quota exceeded)

def add_with_retry(vs, docs):
    """Fuegt ein Batch ein; bei transienten Fehlern (429/5xx) mit Backoff wiederholt."""
    transient = ("429", "500", "502", "503", "504", "RESOURCE_EXHAUSTED", "Bad Gateway", "getaddrinfo", "ConnectError", "timed out")
    for attempt in range(1, MAX_TRIES + 1):
        try:
            vs.add_documents(docs)
            return
        except Exception as e:
            msg = str(e)
            if any(code in msg for code in transient) and attempt < MAX_TRIES:
                wait = 30 * attempt
                print(f"   Transienter Fehler -> warte {wait}s (Versuch {attempt}/{MAX_TRIES - 1})...")
                time.sleep(wait)
            else:
                raise

# Idempotenz: Bestehende Collection loeschen, damit mehrfaches Ausfuehren keine
# Duplikate erzeugt (ueber die Chroma-API statt shutil.rmtree = robuster unter
# Windows, da Chroma die Index-Dateien sonst exklusiv sperrt).
_purge = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)
try:
    _purge.delete_collection()
    print("   Bestehende Collection geloescht.")
except Exception:
    print("   Keine bestehende Collection – frischer Aufbau.")

# Frische (persistente) Collection anlegen und chargenweise fuellen
vectorstore = Chroma(embedding_function=embeddings, persist_directory=CHROMA_DIR)

total = len(chunks)
for start in range(0, total, BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    add_with_retry(vectorstore, batch)
    done = min(start + BATCH_SIZE, total)
    print(f"   Eingefuegt: {done}/{total} Chunks")

print(f"\nVektordatenbank gespeichert unter: {CHROMA_DIR}")
print(f"Indexierte Vektoren: {vectorstore._collection.count()}")

## 5) TEST QUERY – Aehnlichkeitssuche (Microservices)
Ein Architektur-bezogener Test-Query gegen die Vektordatenbank. Die Top-3-Treffer
werden zusammen mit ihrer Quell-PDF (Metadaten) ausgegeben.

Hinweis: Die DB ist bereits persistiert – fuer spaetere Suchen kann sie ohne erneutes
Embedding geladen werden:
```python
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
```

In [ ]:
query = "What are the benefits of a microservices architecture?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source", "unbekannt")
    page = doc.metadata.get("page", "?")
    print(f"--- Ergebnis {i} | Quelle: {source} | Seite: {page} ---")
    print(doc.page_content[:300].strip())
    print()